In [1]:
!pip install open3d
!pip install plyfile

  Using cached plyfile-1.1.4-py3-none-any.whl.metadata (43 kB)
Using cached plyfile-1.1.4-py3-none-any.whl (36 kB)


In [5]:
!pip install plyfile torch-geometric pandas scikit-learn tabulate tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 76.8 MB/s eta 0:00:00


In [ ]:
import os
import glob
import json
import random
import sys
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from plyfile import PlyData
from torch_geometric.data import Data, Dataset, DataLoader
from torch_geometric.nn import global_max_pool
from sklearn.metrics import classification_report, confusion_matrix
from tqdm import tqdm

# ==============================================================================
# КОНФИГУРАЦИЯ И ГИПЕРПАРАМЕТРЫ
# ==============================================================================
CONFIG = {
    "seed": 42,
    "num_classes": 20,
    "epochs": 15,
    "batch_size": 16,
    "lr": 0.002,
    "num_points": 2048,
    "feature_dim": 6,
    "runs": 3,
    "classes": [
        "unknown", "pipe", "wire", "wall", "floor", "ceiling", "machine", "desk",
        "rack", "boiler", "conveyor", "structure", "infrastructure", "roof",
        "window", "door", "gate", "terrain", "facade"
    ]
}

CLASS_TO_IDX = {cls: idx for idx, cls in enumerate(CONFIG["classes"])}

def fix_random_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# ==============================================================================
# МАКСИМАЛЬНО ОПТИМИЗИРОВАННЫЙ С-ПАРСЕР ЧЕРЕЗ PANDAS (В 50-80 РАЗ БЫСТРЕЕ)
# ==============================================================================
def process_single_file(args):
    """Парсит ASCII PLY файл через C-движок Pandas, минуя медленный plyfile"""
    file_path, cache_file, class_to_idx = args

    if os.path.exists(cache_file):
        return cache_file

    try:
        header_lines = 0
        with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
            for line in f:
                header_lines += 1
                if "end_header" in line:
                    break


        column_names = [
            'x', 'y', 'z', 'label', 'instance_id',
            'red', 'green', 'blue', 'station_index',
            'circle_index', 'elevation_deg'
        ]

        df = pd.read_csv(
            file_path,
            skiprows=header_lines,
            sep=r'\s+',
            names=column_names,
            engine='c',
            dtype={
                'x': np.float32, 'y': np.float32, 'z': np.float32,
                'red': np.float32, 'green': np.float32, 'blue': np.float32
            }
        )

        x = df['x'].values
        y = df['y'].values
        z = df['z'].values

        labels_raw = df['label'].values
        if labels_raw.dtype.kind in {'U', 'S', 'O'}:
            labels = np.array([class_to_idx.get(str(l), 0) for l in labels_raw], dtype=np.int64)
        else:
            labels = labels_raw.astype(np.int64)

        r = df['red'].values / 255.0
        g = df['green'].values / 255.0
        b = df['blue'].values / 255.0

        payload = {
            'pos': np.vstack([x, y, z]).T,
            'features': np.vstack([x, y, z, r, g, b]).T,
            'labels': labels
        }

        torch.save(payload, cache_file)
        return cache_file

    except Exception as e:
        return f"ERROR: {file_path} -> {e}"


# ==============================================================================
# ДАТАСЕТ С МНОГОПОТОЧНЫМ КЭШИРОВАНИЕМ НА ВСЕХ ЯДРАХ CPU
# ==============================================================================
class FastLidarDataset(Dataset):
    def __init__(self, root_dir, folder_list, cache_dir, num_points=2048, desc=""):
        super().__init__(None, None)
        self.num_points = num_points
        self.data_list = []

        os.makedirs(cache_dir, exist_ok=True)

        raw_file_paths = []
        for folder in folder_list:
            search_path = os.path.join(root_dir, folder, "**", "*.ply")
            raw_file_paths.extend(glob.glob(search_path, recursive=True))

        print(f"[{desc}]: Запуск параллельного кэширования для {len(raw_file_paths)} файлов...")

        tasks = []
        for idx, file_path in enumerate(raw_file_paths):
            cache_file = os.path.join(cache_dir, f"scan_{idx}.pt")
            tasks.append((file_path, cache_file, CLASS_TO_IDX))

        max_workers = os.cpu_count()
        print(f"Задействовано ядер процессора: {max_workers}")

        with ProcessPoolExecutor(max_workers=max_workers) as executor:
            futures = [executor.submit(process_single_file, task) for task in tasks]

            for future in tqdm(as_completed(futures), total=len(futures), desc=f"Кэширование {desc}"):
                result = future.result()
                if result and not result.startswith("ERROR"):
                    self.data_list.append(result)
                else:
                    print(f"\n {result}")

    def len(self):
        return len(self.data_list)

    def get(self, idx):
        cache_data = torch.load(self.data_list[idx], weights_only=False)
        pos = cache_data['pos']
        features = cache_data['features']
        labels = cache_data['labels']

        num_actual_points = pos.shape[0]
        if num_actual_points >= self.num_points:
            choice = np.random.choice(num_actual_points, self.num_points, replace=False)
        else:
            choice = np.random.choice(num_actual_points, self.num_points, replace=True)

        return Data(
            pos=torch.from_numpy(pos[choice]).float(),
            x=torch.from_numpy(features[choice]).float(),
            y=torch.from_numpy(labels[choice]).long()
        )


# ==============================================================================
# АРХИТЕКТУРА МОДЕЛИ
# ==============================================================================
class PointNet2Segmentation(nn.Module):
    def __init__(self, num_classes, in_channels=6):
        super().__init__()
        self.fc1 = nn.Linear(in_channels, 64)
        self.fc2 = nn.Linear(64, 128)
        self.fc3 = nn.Linear(128, 512)

        self.seg_fc1 = nn.Linear(512 + 64, 256)
        self.seg_fc2 = nn.Linear(256, 128)
        self.seg_fc3 = nn.Linear(128, num_classes)

        self.bn1 = nn.BatchNorm1d(64)
        self.bn2 = nn.BatchNorm1d(128)
        self.bn3 = nn.BatchNorm1d(512)
        self.s_bn1 = nn.BatchNorm1d(256)
        self.s_bn2 = nn.BatchNorm1d(128)

    def forward(self, data):
        x, batch = data.x, data.batch
        x_local = F.relu(self.bn1(self.fc1(x)))
        x = F.relu(self.bn2(self.fc2(x_local)))
        x = F.relu(self.bn3(self.fc3(x)))

        x_global = global_max_pool(x, batch)
        batch_counts = torch.bincount(batch)
        x_global_expanded = torch.repeat_interleave(x_global, batch_counts, dim=0)

        x_combined = torch.cat([x_local, x_global_expanded], dim=-1)
        x = F.relu(self.s_bn1(self.seg_fc1(x_combined)))
        x = F.relu(self.s_bn2(self.seg_fc2(x)))
        logits = self.seg_fc3(x)

        return logits

# ==============================================================================
# ЦИКЛЫ ОБУЧЕНИЯ И ТЕСТИРОВАНИЯ
# ==============================================================================
def train_epoch(model, loader, optimizer, criterion, device, epoch_idx, total_epochs):
    model.train()
    total_loss = 0

    progress_bar = tqdm(loader, desc=f"  Эпоха {epoch_idx:02d}/{total_epochs}", leave=False, unit="батч")
    for data in progress_bar:
        data = data.to(device)
        optimizer.zero_grad()
        out = model(data)
        loss = criterion(out, data.y)
        loss.backward()
        optimizer.step()

        current_loss = loss.item()
        total_loss += current_loss
        progress_bar.set_postfix(loss=f"{current_loss:.4f}")

    return total_loss / len(loader)

@torch.no_grad()
def evaluate_model(model, loader, device):
    model.eval()
    all_preds = []
    all_targets = []

    for data in tqdm(loader, desc="  Тестирование модели", leave=False, unit="батч"):
        data = data.to(device)
        out = model(data)
        preds = out.argmax(dim=-1)
        all_preds.extend(preds.cpu().numpy())
        all_targets.extend(data.y.cpu().numpy())

    return np.array(all_targets), np.array(all_preds)

# ==============================================================================
# ОСНОВНОЙ КОНВЕЙЕР ЭКСПЕРИМЕНТА
# ==============================================================================
if __name__ == "__main__":
    DATASET_ROOT = "/content/drive/MyDrive/mipt_tsitis_cv_26/IS"
    OUTPUT_DIR = "/content/drive/MyDrive/mipt_tsitis_cv_26/lab3_results"
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Вычисления запущены на: {device}\n")

    with open(os.path.join(OUTPUT_DIR, "config.json"), "w") as f:
        json.dump(CONFIG, f, indent=4)

    all_scenes = sorted([d for d in os.listdir(DATASET_ROOT) if os.path.isdir(os.path.join(DATASET_ROOT, d)) and not d.startswith("_")])

    if len(all_scenes) >= 3:
        train_folders = all_scenes[:-1]
        test_folders = all_scenes[-1:]
    else:
        train_folders = all_scenes
        test_folders = all_scenes

    print(f"Сцены Обучения: {train_folders} | Сцены Теста: {test_folders}\n")

    # ==============================================================================
    # ПЕРЕНЕСЕННЫЙ ЛОКАЛЬНЫЙ КЭШ ДЛЯ ИСПРАВЛЕНИЯ ОШИБКИ READ-ONLY FILE SYSTEM
    # ==============================================================================
    LOCAL_CACHE_DIR = "/content/drive/MyDrive/mipt_tsitis_cv_26/lab3_results/lidar_fast_cache"
    os.makedirs(LOCAL_CACHE_DIR, exist_ok=True)

    print("=== ЭТАП 1: Инициализация и кэширование датасетов ===")

    train_dataset = FastLidarDataset(
        root_dir=DATASET_ROOT,
        folder_list=train_folders,
        cache_dir=os.path.join(LOCAL_CACHE_DIR, "TRAIN"),
        num_points=CONFIG["num_points"],
        desc="TRAIN"
    )
    test_dataset = FastLidarDataset(
        root_dir=DATASET_ROOT,
        folder_list=test_folders,
        cache_dir=os.path.join(LOCAL_CACHE_DIR, "TEST"),
        num_points=CONFIG["num_points"],
        desc="TEST"
    )

    num_workers = 2 if os.name != 'nt' else 0

    train_loader = DataLoader(
        train_dataset,
        batch_size=CONFIG["batch_size"],
        shuffle=True,
        num_workers=num_workers,
        pin_memory=True
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=CONFIG["batch_size"],
        shuffle=False,
        num_workers=num_workers,
        pin_memory=True
    )

    print("\n=== ЭТАП 2: Запуск глубокого обучения ===")
    run_metrics = []

    for run_id in range(1, CONFIG["runs"] + 1):
        print(f"СТАРТ НЕЗАВИСИМОГО ЗАПУСКА №{run_id}/{CONFIG['runs']}")

        fix_random_seeds(CONFIG["seed"] + run_id)

        model = PointNet2Segmentation(num_classes=CONFIG["num_classes"], in_channels=CONFIG["feature_dim"]).to(device)
        optimizer = torch.optim.Adam(model.parameters(), lr=CONFIG["lr"])
        criterion = nn.CrossEntropyLoss()

        for epoch in range(1, CONFIG["epochs"] + 1):
            avg_loss = train_epoch(model, train_loader, optimizer, criterion, device, epoch, CONFIG["epochs"])
            if epoch % 5 == 0 or epoch == CONFIG["epochs"]:
                print(f"    -> Итог эпохи {epoch:02d}: Средний Loss = {avg_loss:.4f}")

        targets, preds = evaluate_model(model, test_loader, device)

        cm = confusion_matrix(targets, preds, labels=list(range(CONFIG["num_classes"])))
        np.save(os.path.join(OUTPUT_DIR, f"confusion_matrix_run_{run_id}.npy"), cm)

        report = classification_report(
            targets, preds,
            labels=list(range(CONFIG["num_classes"])),
            target_names=CONFIG["classes"],
            output_dict=True,
            zero_division=0
        )

        intersection = np.diag(cm)
        union = cm.sum(axis=1) + cm.sum(axis=0) - intersection
        iou = np.where(union > 0, intersection / union, 0)
        miou = np.mean(iou[union > 0])

        oa = report["accuracy"]
        macro_f1 = report["macro avg"]["f1-score"]

        run_metrics.append({
            "run": run_id,
            "OA": float(oa),
            "Macro_F1": float(macro_f1),
            "mIoU": float(miou),
            "per_class_iou": iou.tolist(),
            "report": report
        })

        torch.save(model.state_dict(), os.path.join(OUTPUT_DIR, f"model_weights_run_{run_id}.pth"))
        print(f"Запуск №{run_id} завершен. OA: {oa:.4f} | mIoU: {miou:.4f}\n")

    # ==============================================================================
    # СВЕДЕНИЕ ОТЧЕТНОСТИ И МАТЕМАТИЧЕСКИЙ АНАЛИЗ (Усреднение результатов и разброс)
    # ==============================================================================
    df_res = pd.DataFrame(run_metrics)
    print("\n" + "="*70 + "\n ИТОГОВЫЙ СТАТИСТИЧЕСКИЙ АНАЛИЗ СИСТЕМЫ\n" + "="*70)

    summary_table = []
    for metric in ["OA", "Macro_F1", "mIoU"]:
        values = df_res[metric].values
        summary_table.append([
            metric,
            f"{np.mean(values):.5f}",
            f"±{np.std(values):.5f}",
            f"{np.min(values):.5f}",
            f"{np.max(values):.5f}"
        ])

    from tabulate import tabulate
    print("\n[ГЛОБАЛЬНЫЕ МЕТРИКИ КАЧЕСТВА СЕГМЕНТАЦИИ]:")
    print(tabulate(summary_table, headers=["Метрика", "Среднее значение", "Разброс (Std)", "Min", "Max"], tablefmt="grid"))

    print("\n[ПОКЛАССОВЫЙ АНАЛИЗ ТАБЛИЦЫ (УСРЕДНЕННЫЙ ПО 3 ЗАПУСКАМ)]:")
    per_class_summary = []
    for idx, class_name in enumerate(CONFIG["classes"]):
        p = [r["report"][class_name]["precision"] for r in run_metrics]
        r = [r["report"][class_name]["recall"] for r in run_metrics]
        f = [r["report"][class_name]["f1-score"] for r in run_metrics]
        i = [r["per_class_iou"][idx] for r in run_metrics]
        per_class_summary.append([
            class_name,
            f"{np.mean(p):.4f}",
            f"{np.mean(r):.4f}",
            f"{np.mean(f):.4f}",
            f"{np.mean(i):.4f}"
        ])
    print(tabulate(per_class_summary, headers=["Имя класса", "Precision", "Recall", "F1-Score", "IoU"], tablefmt="simple"))

    print("\n[МАТРИЦА ОШИБОК / CONFUSION MATRIX (Итоговый запуск)]:")
    cm_df = pd.DataFrame(cm, index=CONFIG["classes"], columns=CONFIG["classes"])
    pd.set_option('display.max_columns', None)
    pd.set_option('display.width', 1000)
    print(cm_df)

    print(f"\n Эксперимент полностью завершен. Результаты сохранены в папку: '{OUTPUT_DIR}'")



🖥️ Вычисления запущены на: cuda

🎬 Сцены Обучения: ['boliler', 'control', 'electrical', 'laboratory', 'maintenance', 'office', 'refinery'] | Сцены Теста: ['storage']

=== ЭТАП 1: Инициализация и кэширование датасетов ===
📦 [TRAIN]: Запуск параллельного кэширования для 582 файлов...
⚙️ Задействовано ядер процессора: 2


Кэширование TRAIN: 100%|██████████| 582/582 [00:00<00:00, 2961.02it/s]


📦 [TEST]: Запуск параллельного кэширования для 129 файлов...
⚙️ Задействовано ядер процессора: 2


Кэширование TEST: 100%|██████████| 129/129 [00:00<00:00, 1840.93it/s]
/tmp/ipykernel_45307/540116067.py:310: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  train_loader = DataLoader(
/tmp/ipykernel_45307/540116067.py:318: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  test_loader = DataLoader(



=== ЭТАП 2: Запуск глубокого обучения ===
🚀 СТАРТ НЕЗАВИСИМОГО ЗАПУСКА №1/3


...

    -> Итог эпохи 10: Средний Loss = 0.0646


    -> Итог эпохи 15: Средний Loss = 0.0318


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:2687: UserWarning: labels size, 20, does not match size of target_names, 19
  warnings.warn(
/tmp/ipykernel_45307/540116067.py:366: RuntimeWarning: invalid value encountered in divide
  iou = np.where(union > 0, intersection / union, 0)


✨ Запуск №3 завершен. OA: 0.4730 | mIoU: 0.2948


 📊 ИТОГОВЫЙ СТАТИСТИЧЕСКИЙ АНАЛИЗ СИСТЕМЫ

[ГЛОБАЛЬНЫЕ МЕТРИКИ КАЧЕСТВА СЕГМЕНТАЦИИ]:
+-----------+--------------------+-----------------+---------+---------+
| Метрика   |   Среднее значение | Разброс (Std)   |     Min |     Max |
+===========+====================+=================+=========+=========+
| OA        |            0.61267 | ±0.09879        | 0.47296 | 0.68282 |
+-----------+--------------------+-----------------+---------+---------+
| Macro_F1  |            0.14117 | ±0.02233        | 0.1096  | 0.1576  |
+-----------+--------------------+-----------------+---------+---------+
| mIoU      |            0.35479 | ±0.04461        | 0.29481 | 0.4017  |
+-----------+--------------------+-----------------+---------+---------+

[ПОКЛАССОВЫЙ АНАЛИЗ ТАБЛИЦЫ (УСРЕДНЕННЫЙ ПО 3 ЗАПУСКАМ)]:
Имя класса        Precision    Recall    F1-Score     IoU
--------------  -----------  --------  ----------  ------
unknown              0.2461    0

ValueError: Shape of passed values is (20, 20), indices imply (19, 19)

In [ ]:
    # ==============================================================================
    # ИНИЦИАЛИЗАЦИЯ ДАННЫХ ПОД СТРУКТУРУ ФОЛДЕРОВ (Без смешивания сцен)
    # ==============================================================================
    DATASET_ROOT = "/content/drive/MyDrive/mipt_tsitis_cv_26/IS"
    OUTPUT_DIR = "/content/drive/MyDrive/mipt_tsitis_cv_26/lab3_results"
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    with open(os.path.join(OUTPUT_DIR, "config.json"), "w") as f:
        json.dump(CONFIG, f, indent=4)

    all_scenes = sorted([
        d for d in os.listdir(DATASET_ROOT)
        if os.path.isdir(os.path.join(DATASET_ROOT, d))
    ])

    if len(all_scenes) >= 3:
        train_folders = all_scenes[:-1]
        test_folders = all_scenes[-1:]
    else:
        train_folders = all_scenes
        test_folders = all_scenes
        print("Слишком мало верхнеуровневых сцен. Разделение будет произведено внутри датасета.")

    print(f"Сцены для обучения (Train): {train_folders}")
    print(f"Сцены для проверки (Test): {test_folders}")

    run_metrics = []
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


🎬 Сцены для обучения (Train): ['boliler', 'control', 'electrical', 'laboratory', 'maintenance', 'office', 'refinery']
🎯 Сцены для проверки (Test): ['storage']


In [ ]:
extended_classes = CONFIG["classes"] + ["extra_class"] if len(CONFIG["classes"]) == 19 else CONFIG["classes"]

print("\n[МАТРИЦА ОШИБОК / CONFUSION MATRIX (Итоговый запуск)]:")
cm_df = pd.DataFrame(cm, index=extended_classes, columns=extended_classes)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.width', 1000)
print(cm_df)



[МАТРИЦА ОШИБОК / CONFUSION MATRIX (Итоговый запуск)]:
                unknown  pipe  wire   wall  floor  ceiling  machine  desk  rack  boiler  conveyor  structure  infrastructure  roof  window  door  gate  terrain  facade  extra_class
unknown            7586     0     0   3115      0        0        0     0     0       0         0          0               0     0       0     0     0        0       0            0
pipe                  0     0     0      0      0        0        0     0     0       0         0          0               0     0       0     0     0        0       0            0
wire                  0     0     0      0      0        0        0     0     0       0         0          0               0     0       0     0     0        0       0            0
wall                  0     0     0  29837      0        0        0     0     0       0         0          0               0     0       0     0     0        0       0            0
floor                 0     0     0    